# 02 - Dataset e Labels

## O que esta fase faz

Labels JSON foram gerados com GPT-4o, validados no Label Studio e exportados para SFT (`data/sft/*.jsonl`).

**Não é necessário recriar labels** para a entrega do TCC.

## Schema JSON

```json
{
  "leitura": {"inteiro": 302, "decimal": 21, "completo": "0302,21"},
  "fabricante": "Saga",
  "estado": "normal"
}
```

- **completo** - transcrição literal (métrica CER)
- **inteiro/decimal** - valores numéricos dos roletes

In [1]:
from __future__ import annotations
import json, os, sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "src" / "hidrometro").exists()), None)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import display
from hidrometro.config import resolve_path, load_yaml

sft_dir = resolve_path(load_yaml("paths.yaml")["output"]["sft"])
counts = {}
for split in ["train", "val", "test"]:
    p = sft_dir / f"{split}.jsonl"
    counts[split] = sum(1 for line in p.read_text(encoding="utf-8").splitlines() if line.strip()) if p.exists() else 0
display(pd.DataFrame([{"split": k, "amostras": v} for k, v in counts.items()]))

,split,amostras
0,train,503
1,val,144
2,test,72


In [2]:
audit_path = resolve_path("reports") / "label_audit.json"
if not audit_path.exists():
    import subprocess
    subprocess.run([sys.executable, "scripts/06_audit_labels.py"], cwd=ROOT, check=False)

if audit_path.exists():
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    print(f"Consistência inteiro/completo: {audit['consistency_rate']*100:.1f}%")
    print(f"Inconsistentes: {audit['inconsistent']} / {audit['total_records']}")
    if audit["issues"]:
        display(pd.DataFrame(audit["issues"][:10]))
else:
    print("Rode: python scripts/06_audit_labels.py")

Consistência inteiro/completo: 100.0%
Inconsistentes: 0 / 719


## Exemplo de registro SFT

**Próximo notebook:** `03_treino_qlora.ipynb` - processo de fine-tuning.

In [3]:
sample = json.loads((sft_dir / "train.jsonl").read_text(encoding="utf-8").splitlines()[0])
print(json.dumps({k: sample[k] for k in ["prompt", "response"]}, ensure_ascii=False, indent=2))

{
  "prompt": "<VQA>Leia o hidrômetro e retorne JSON com leitura (inteiro, decimal, completo), fabricante e estado.",
  "response": "{\"leitura\": {\"inteiro\": 7, \"decimal\": 67, \"completo\": \"0007,67\"}, \"fabricante\": \"ENERGYRUS\", \"estado\": \"normal\"}"
}
